# Building a RAG App 

This notebook implements a RAG system following Langchain's official documentation, adapted to use open source models:

- **LLM**: LLama via Ollama (not using any type of Gemini or OpenAI models)
- **Embeddings**: Sentence Transformers locally 
- **VectorStore**: InMemoryVectorStore (same as the documentation for now, but it can be better by using a ChromaDB)

## Overview

A typical RAG application have 2 principal components:
1. **Indexing**: pipeline to ingest data from a source and index them
2. **Retrieval and Generation**: RAG chain that gets the user query/input and fetches relevant data from the index

Basically being the flow:

**Store** -> **Retrieve** -> **Generate**

# Setup and Installation

Firstly, you are going to install the necessary dependencies to our RAG system

In [1]:
%pip install --quiet --upgrade langchain-text-splitters langchain-community langgraph
%pip install --quiet --upgrade langchain-ollama sentence-transformers
%pip install --quiet --upgrade beautifulsoup4 requests

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


# Components

Let's go to select 3 principal components to this notebook, which are:

1. **Chat Model**: LLama via Ollama
2. **Embeddings Model**: Sentence Transformers (local)
3. **Vector Store**: InMemoryVectorStore (simple and efficient)


In [2]:
import os
from langchain_ollama.chat_models import ChatOllama

os.environ["OLLAMA_BASE_URL"] = "http://localhost:11434"

llm = ChatOllama(
    model="llama3.2",
    temperature=0.1,
    base_url="http://localhost:11434",
)

print(f"Model: {llm.model}")
print(f"Temperature: {llm.temperature}")

Model: llama3.2
Temperature: 0.1


In [3]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

print(f"Embedding model: {embeddings.model_name}")
print(f"Embedding device: {embeddings.model_kwargs['device']}")

/tmp/ipykernel_4756/1145482280.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/home/joao/.cache/pypoetry/virtualenvs/roshi-rag-kyW4OOdz-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding device: cpu


In [4]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

print("MemoryStore configured.")

MemoryStore configured.


# Preview

Now I am going to create a simple RAG pipeline that answers questions about a specific website

Following Langchain's documentation accordingly, let's use the blog post "LLM Powered Autonomous Agents" from Lilian Weng as the source to demonstrate how does RAG function at all.

The complete pipeline will be implemented and include:
- **Load** the source data (website)
- **Split** the data in small chunks
- **Store** it in the vector store
- **Retrieve** relevant documents
- **Generate** answers based on the documents context

In [5]:
from typing import List, TypedDict
import bs4
from langchain import hub
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langgraph.graph import START, StateGraph

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs)

# Index chunks in the vector store
print(f"Indexing {len(all_splits)} chunks in the vector store...")
_ = vector_store.add_documents(documents=all_splits)

# Define prompt for question-answering
# Pulling the prompt from the hub, but you can also define it directly
# as a string if you prefer. Or in Langsmith or Langfuse.
prompt = hub.pull("rlm/rag-prompt")

print("RAG prompt loaded from LangChain Hub")
print(f"Prompt type: {type(prompt)}")

# Define the state for the RAG agent
class State(TypedDict):
    question: str
    context: List[Document]
    answer: str

def retrieve(state: State):
    retrieved_docs = vector_store.similarity_search(state["question"])
    return {"context": retrieved_docs}

def generate(state: State):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = prompt.invoke({"question": state["question"], "context": docs_content})
    response = llm.invoke(messages)
    return {"answer": response.content}

graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

USER_AGENT environment variable not set, consider setting it to identify your requests.


Indexing 63 chunks in the vector store...


/home/joao/.cache/pypoetry/virtualenvs/roshi-rag-kyW4OOdz-py3.12/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/home/joao/.cache/pypoetry/virtualenvs/roshi-rag-kyW4OOdz-py3.12/lib/python3.12/site-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


RAG prompt loaded from LangChain Hub
Prompt type: <class 'langchain_core.prompts.chat.ChatPromptTemplate'>


In [6]:
response = graph.invoke({"question": "What is an agent?"})
print(response["answer"])

/home/joao/.cache/pypoetry/virtualenvs/roshi-rag-kyW4OOdz-py3.12/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


An agent is an entity that perceives its environment, takes actions, and learns from the consequences of those actions to achieve a goal or set of goals. In the context of LLM-powered autonomous agents, the LLM serves as the brain, making decisions based on planning, reflection, and refinement, while also relying on memory to store information and learn from past experiences. The agent's primary objective is to optimize its believability in the moment versus long-term.
